In [5]:
import pandas as pd
import numpy as np
import os
import math

# Configuration
TARGET_STEPS = 5000 
DOWNSAMPLE_FACTOR = 4  # Force 200Hz -> 50Hz

# Files to process
file_paths = [
    "/media/kai/NewDisk/Kai_thesis/Master_Thesis_E2E_RL_Teleop/libfranka_ws/src/E2E_Teleoperation/E2E_Teleoperation/evaluation/PD_controller/high_delay.xlsx",
    "/media/kai/NewDisk/Kai_thesis/Master_Thesis_E2E_RL_Teleop/libfranka_ws/src/E2E_Teleoperation/E2E_Teleoperation/evaluation/PD_controller/high_variance.xlsx",
    "/media/kai/NewDisk/Kai_thesis/Master_Thesis_E2E_RL_Teleop/libfranka_ws/src/E2E_Teleoperation/E2E_Teleoperation/evaluation/PD_controller/low_delay.xlsx"
]

# Mapping from PD headers (Raw) to Standard headers
COLUMN_MAPPING = {
    'leader_ee_pose_point.x': 'leader_ee_pos_x',
    'leader_ee_pose_point.y': 'leader_ee_pos_y',
    'leader_ee_pose_point.z': 'leader_ee_pos_z',
    'follower_ee_pose_point.x': 'follower_ee_pos_x',
    'follower_ee_pose_point.y': 'follower_ee_pos_y',
    'follower_ee_pose_point.z': 'follower_ee_pos_z'
}

def augment_data_pattern_repeat(df, target_length):
    """
    Extends the dataframe to `target_length` by repeating the ENTIRE existing pattern
    (cyclic repetition) instead of just the last value.
    """
    current_length = len(df)
    
    if current_length >= target_length:
        return df.head(target_length)
    
    # Calculate how many times we need to repeat the data
    repeat_factor = math.ceil(target_length / current_length)
    
    print(f"    -> Augmenting data: Pattern length {current_length} repeated {repeat_factor} times...")
    
    # Repeat the dataframe
    df_repeated = pd.concat([df] * repeat_factor, ignore_index=True)
    
    # Truncate to exact target length
    df_final = df_repeated.head(target_length)
    
    return df_final

def process_pd_data(file_path):
    if not os.path.exists(file_path):
        print(f"Error: File not found at {file_path}")
        return

    try:
        print(f"Processing {os.path.basename(file_path)}...")
        
        # 1. Load Excel
        df = pd.read_excel(file_path, engine='openpyxl')
        
        # 2. Rename Columns
        df.rename(columns=COLUMN_MAPPING, inplace=True)
        
        # Verify renaming success
        required_cols = ['leader_ee_pos_x', 'follower_ee_pos_x']
        if not all(col in df.columns for col in required_cols):
            print(f"  - Warning: Column renaming failed. Available columns: {df.columns.tolist()}")
            return

        # 3. Downsample (200Hz -> 50Hz)
        df_downsampled = df.iloc[::DOWNSAMPLE_FACTOR].reset_index(drop=True)
        print(f"  - Downsampled shape: {df_downsampled.shape}")

        # 4. Augment Pattern
        df_final = augment_data_pattern_repeat(df_downsampled, TARGET_STEPS)
        print(f"  - Final shape: {df_final.shape}")

        # 5. Save as CSV
        file_dir, file_name = os.path.split(file_path)
        name_root, _ = os.path.splitext(file_name)
        output_path = os.path.join(file_dir, f"{name_root}_cleaned.csv")
        
        df_final.to_csv(output_path, index=False)
        print(f"  - Saved to: {output_path}\n")

    except Exception as e:
        print(f"An error occurred: {e}\n")

if __name__ == "__main__":
    print("Starting PD Data Cleaning & Pattern Repetition...\n")
    for path in file_paths:
        process_pd_data(path)
    print("Processing Complete.")

Starting PD Data Cleaning & Pattern Repetition...

Processing high_delay.xlsx...
  - Downsampled shape: (600, 8)
    -> Augmenting data: Pattern length 600 repeated 9 times...
  - Final shape: (5000, 8)
  - Saved to: /media/kai/NewDisk/Kai_thesis/Master_Thesis_E2E_RL_Teleop/libfranka_ws/src/E2E_Teleoperation/E2E_Teleoperation/evaluation/PD_controller/high_delay_cleaned.csv

Processing high_variance.xlsx...
  - Downsampled shape: (600, 8)
    -> Augmenting data: Pattern length 600 repeated 9 times...
  - Final shape: (5000, 8)
  - Saved to: /media/kai/NewDisk/Kai_thesis/Master_Thesis_E2E_RL_Teleop/libfranka_ws/src/E2E_Teleoperation/E2E_Teleoperation/evaluation/PD_controller/high_variance_cleaned.csv

Processing low_delay.xlsx...
  - Downsampled shape: (601, 8)
    -> Augmenting data: Pattern length 601 repeated 9 times...
  - Final shape: (5000, 8)
  - Saved to: /media/kai/NewDisk/Kai_thesis/Master_Thesis_E2E_RL_Teleop/libfranka_ws/src/E2E_Teleoperation/E2E_Teleoperation/evaluation/PD_c